# IOGraphProcessor demo

This notebook demonstrates [`IOGraphProcessor`](../../src/phopymnehelper/iograph_data.py) from `phopymnehelper`.

IOGraphica exports mouse-position CSV files whose filenames encode approximate session start/end times. The processor:

1. Scans a directory of CSV exports and parses filename time ranges
2. Merges overlapping exports into sessions (incremental saves deduplicated)
3. Builds a single `master_df` with absolute `sample_time` timestamps
4. Exposes timeline-ready `detail_df` (unix `t`, `x`, `y`) and `intervals_df` views

In [ ]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3
from datetime import datetime
from pathlib import Path

import pandas as pd

from phopymnehelper.iograph_data import IOGraphProcessor

In [ ]:
# Point this at a folder of IOGraphica CSV exports.
iograph_directory = Path(r"C:\Users\pho\Desktop\IOGraphOutputs").resolve()
assert iograph_directory.is_dir(), f"IOGraph directory not found: {iograph_directory}"

timestamp_str = datetime.now().strftime("%Y-%m-%d_%I%M%p").lower()
output_dir = iograph_directory.parent

## One-shot load: `from_directory`

The simplest entry point scans the folder, merges sessions, and populates `file_table_df` and `master_df`.

In [ ]:
processor = IOGraphProcessor.from_directory(iograph_directory, recursive=False, drop_na_coords=True)

print(f"Scanned {len(processor.file_table_df):,} CSV file(s)")
print(f"Built master_df with {len(processor.master_df):,} sample row(s)")
print(f"Sessions: {processor.master_df['session_index'].nunique() if not processor.master_df.empty else 0}")

In [ ]:
processor.file_table_df

In [ ]:
master_df = processor.master_df

if not master_df.empty:
    end_check = master_df.groupby("session_index").agg(
        sample_time_max=("sample_time", "max"),
        parsed_filename_dt_end=("parsed_filename_dt_end", "max"),
    )
    end_check["end_delta_sec"] = (
        end_check["sample_time_max"] - end_check["parsed_filename_dt_end"]
    ).dt.total_seconds()
    assert end_check["end_delta_sec"].abs().max() < 1e-3
    assert master_df["session_index"].is_monotonic_increasing
    assert not master_df.duplicated(subset=["session_index", "sample_time"]).any()

master_df.head()

In [ ]:
new_cols_df, intervals_df = IOGraphProcessor.compute_psychomotor_metrics(detail_df=processor.detail_df)
intervals_df

## 11m 54s -- len(intervals_df): 107919 rows, len(processor.detail_df): 7443143, len(new_cols_df): 7443143

## 18.7s -- len(intervals_df): 107919 rows, len(processor.detail_df): 7443143, len(new_cols_df): 7443143

In [ ]:
new_cols_df

In [ ]:
master_df: pd.DataFrame = processor.master_df
master_df

In [ ]:
# len(processor.detail_df)
detail_df: pd.DataFrame = processor.detail_df
detail_df


## Timeline views

`detail_df` and `intervals_df` are derived from `master_df` and match what `IOGraphRecordingsTrackDatasource` uses in pyPhoTimeline.

In [ ]:
detail_df = processor.detail_df
intervals_df = processor.intervals_df

print("detail_df columns:", list(detail_df.columns))
print("intervals_df columns:", list(intervals_df.columns))

display(detail_df.head())
display(intervals_df)

## Step-by-step API

You can also call the class methods directly when you need more control (for example, to write intermediate CSVs).

In [ ]:
file_table_path = output_dir / f"IOGraphOutputs_file_table_{timestamp_str}.csv"
master_path = output_dir / f"IOGraphOutputs_master_{timestamp_str}.csv"

file_table_df, _ = IOGraphProcessor.scan_csv_directory(
    iograph_directory,
    output_csv=file_table_path,
    recursive=False,
)
master_df_stepwise, _ = IOGraphProcessor.build_master_df(
    file_table_df,
    output_csv=master_path,
    drop_na_coords=True,
)

stepwise_processor = IOGraphProcessor(
    directory=iograph_directory,
    file_table_df=file_table_df,
    master_df=master_df_stepwise,
)

print(f"Wrote file table to {file_table_path}")
print(f"Wrote master df to {master_path}")
pd.testing.assert_frame_equal(
    processor.master_df.reset_index(drop=True),
    stepwise_processor.master_df.reset_index(drop=True),
)
stepwise_processor.master_df.tail()

## Save from an existing processor instance

In [ ]:
saved_file_table = processor.save_file_table_csv(output_dir / f"IOGraphOutputs_file_table_saved_{timestamp_str}.csv")
saved_master = processor.save_master_csv(output_dir / f"IOGraphOutputs_master_saved_{timestamp_str}.csv")

print(saved_file_table)
print(saved_master)

## Filename time-range parsing

IOGraphica encodes session bounds in filenames. The processor parses both same-day and cross-midnight formats.

In [ ]:
examples = [
    (
        "IOGraphica - 1 hour (from 14-37 to 15-40).csv",
        pd.Timestamp("2026-05-12 19:40:17"),
    ),
    (
        "IOGraphica - 1.3 hours (from 22-56 May 11th to 0-16 May 12th).csv",
        pd.Timestamp("2026-05-12 04:16:17"),
    ),
]

for file_name, modified in examples:
    dt_start, dt_end = IOGraphProcessor._parsed_filename_dt_range(file_name, modified)
    print(f"{file_name}")
    print(f"  start: {dt_start}")
    print(f"  end:   {dt_end}")
    print()